# Logical Fallacy Detection - Progress 1
**Dataset:** CoCoLoFa (Comments with Common Logical Fallacies)
https://github.com/Crowd-AI-Lab/cocolofa

**Progress 1:** Data Preprocessing and Text Representations:
- **TF-IDF** (Term Frequency-Inverse Document Frequency)
- **Word2Vec** (Word embeddings averaged per document)

**Citation:**
Min-Hsuan Yeh, Ruyuan Wan, and Ting-Hao 'Kenneth' Huang. (2024). *CoCoLoFa: A Dataset of News Comments with Common Logical Fallacies Written by LLM-Assisted Crowds*. [arXiv:2410.03457](https://arxiv.org/abs/2410.03457)

## 0. Import Libraries

In [18]:
import json
import re
import os
import numpy as np
import pandas as pd

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import Word2Vec

## 1. Load Dataset

In [19]:
KAGGLE_PATH = '/kaggle/input/datasets/aristelcrowley/cocolofa'

if os.path.exists(KAGGLE_PATH):
    DATA_DIR = KAGGLE_PATH
    print(f"Running on Kaggle (dataset: {DATA_DIR})")
else:
    DATA_DIR = 'data'   
    print("Running locally")

file_path = os.path.join(DATA_DIR, 'train.json')

if not os.path.exists(file_path):
    print(f"Error: Could not find train.json at {file_path}")
else:
    with open(file_path, 'r', encoding='utf-8') as f:
        raw_data = json.load(f)
        
    records = []
    for article in raw_data:
        for comment in article['comments']:
            records.append({
                'comment_id': comment['id'],
                'news_id': comment['news_id'],
                'fallacy': comment['fallacy'],
                'comment': comment['comment']
            })

    df = pd.DataFrame(records)

    print(f"Total komentar: {len(df)}")
    print(f"\nDistribusi label fallacy:")
    print(df['fallacy'].value_counts())
    print(f"\n--- Sample Data (Teks Mentah) ---")
    
    display(df.head(10))

Running locally
Total komentar: 5370

Distribusi label fallacy:
fallacy
none                        2202
slippery slope               431
appeal to worse problems     421
appeal to nature             412
appeal to tradition          401
false dilemma                391
appeal to majority           383
hasty generalization         379
appeal to authority          350
Name: count, dtype: int64

--- Sample Data (Teks Mentah) ---


,comment_id,news_id,fallacy,comment
0,5584,262,none,Lack of transparency in government isn't unexp...
1,5582,262,appeal to authority,While the issues discussed here should be addr...
2,5583,262,none,The excuse that Brazilian municipalities do no...
3,6183,262,none,This is what's to be expected of developing an...
4,6182,262,appeal to tradition,"Sad to say, I have to agree with you. Rulers c..."
5,6184,262,none,Why doesn't the Brazilian federal government s...
6,6782,262,appeal to worse problems,At least they have a set up for it. In some co...
7,6783,262,hasty generalization,If these people can't even manage doing someth...
8,6784,262,appeal to tradition,I completely agree with you! There have alway...
9,10082,262,slippery slope,The argument that people should have access to...


## 2. Text Cleaning

In [20]:
def clean_text(text):
    """Membersihkan teks mentah."""
    text = text.lower()                                    # lowercase
    text = re.sub(r'http\S+|www\.\S+', '', text)           # hapus URL
    text = re.sub(r'<[^>]+>', '', text)                    # hapus HTML tags
    text = re.sub(r'@\w+', '', text)                       # hapus mentions
    text = re.sub(r'#\w+', '', text)                       # hapus hashtags
    text = re.sub(r'[^a-zA-Z\s]', '', text)                # hapus angka & special chars
    text = re.sub(r'\s+', ' ', text).strip()               # hapus spasi berlebih
    return text

df['cleaned'] = df['comment'].apply(clean_text)

print("--- Contoh Sebelum & Sesudah Cleaning ---")
for i in range(3):
    print(f"\n[BEFORE]: {df['comment'].iloc[i][:120]}...")
    print(f"[AFTER ]: {df['cleaned'].iloc[i][:120]}...")

--- Contoh Sebelum & Sesudah Cleaning ---

[BEFORE]: Lack of transparency in government isn't unexpected.  No one likes to be under scrutiny,  especially not those with powe...
[AFTER ]: lack of transparency in government isnt unexpected no one likes to be under scrutiny especially not those with power tra...

[BEFORE]: While the issues discussed here should be addressed, there need to be priorities. The authorities of the country probabl...
[AFTER ]: while the issues discussed here should be addressed there need to be priorities the authorities of the country probably ...

[BEFORE]: The excuse that Brazilian municipalities do not have the funds to handle the public service requests does not hold water...
[AFTER ]: the excuse that brazilian municipalities do not have the funds to handle the public service requests does not hold water...


## 3. Tokenisasi

In [21]:
df['tokens'] = df['cleaned'].apply(word_tokenize)

print("--- Contoh Hasil Tokenisasi ---")
for i in range(3):
    print(f"\n[CLEANED ]: {df['cleaned'].iloc[i][:100]}...")
    print(f"[TOKENS  ]: {df['tokens'].iloc[i][:15]}...")
    print(f"Jumlah token: {len(df['tokens'].iloc[i])}")

--- Contoh Hasil Tokenisasi ---

[CLEANED ]: lack of transparency in government isnt unexpected no one likes to be under scrutiny especially not ...
[TOKENS  ]: ['lack', 'of', 'transparency', 'in', 'government', 'isnt', 'unexpected', 'no', 'one', 'likes', 'to', 'be', 'under', 'scrutiny', 'especially']...
Jumlah token: 64

[CLEANED ]: while the issues discussed here should be addressed there need to be priorities the authorities of t...
[TOKENS  ]: ['while', 'the', 'issues', 'discussed', 'here', 'should', 'be', 'addressed', 'there', 'need', 'to', 'be', 'priorities', 'the', 'authorities']...
Jumlah token: 55

[CLEANED ]: the excuse that brazilian municipalities do not have the funds to handle the public service requests...
[TOKENS  ]: ['the', 'excuse', 'that', 'brazilian', 'municipalities', 'do', 'not', 'have', 'the', 'funds', 'to', 'handle', 'the', 'public', 'service']...
Jumlah token: 93


## 4. Stopword Removal & Lemmatization

> Dipilih **lemmatization** daripada stemming karena menghasilkan kata dasar yang valid secara bahasa, lebih cocok untuk tugas deteksi fallacy yang membutuhkan pemahaman makna.

In [22]:
# Words that are NLTK stopwords but are KEY fallacy indicators — we keep these
# "everyone/always/never/must/should/only/just" → crucial for detecting fallacy types
fallacy_indicators = {
    'everyone', 'always', 'never', 'must', 'should', 'all', 'every',
    'only', 'just', 'no', 'nothing', 'everything', 'most', 'any',
    'either', 'or', 'neither', 'nor', 'both', 'very', 'too',
}
stop_words = set(stopwords.words('english')) - fallacy_indicators
print(f"Stopwords: {len(set(stopwords.words('english')))} total → {len(stop_words)} after keeping {len(fallacy_indicators)} fallacy indicators")

lemmatizer = WordNetLemmatizer()

def lemmatize_tokens(tokens):
    """Hapus stopwords (kecuali fallacy indicators) dan lemmatize setiap token."""
    return [
        lemmatizer.lemmatize(token)
        for token in tokens
        if (token not in stop_words and len(token) > 2) or token in fallacy_indicators
    ]

df['lemmatized'] = df['tokens'].apply(lemmatize_tokens)

# Gabungkan kembali menjadi string untuk keperluan TF-IDF
df['processed_text'] = df['lemmatized'].apply(lambda x: ' '.join(x))

print("\n--- Contoh Hasil Lemmatization ---")
for i in range(3):
    print(f"\n[TOKENS     ]: {df['tokens'].iloc[i][:12]}...")
    print(f"[LEMMATIZED ]: {df['lemmatized'].iloc[i][:12]}...")
    print(f"Jumlah token sebelum: {len(df['tokens'].iloc[i])} → sesudah: {len(df['lemmatized'].iloc[i])}")

Stopwords: 198 total → 186 after keeping 21 fallacy indicators

--- Contoh Hasil Lemmatization ---

[TOKENS     ]: ['lack', 'of', 'transparency', 'in', 'government', 'isnt', 'unexpected', 'no', 'one', 'likes', 'to', 'be']...
[LEMMATIZED ]: ['lack', 'transparency', 'government', 'isnt', 'unexpected', 'no', 'one', 'like', 'scrutiny', 'especially', 'power', 'transparency']...
Jumlah token sebelum: 64 → sesudah: 33

[TOKENS     ]: ['while', 'the', 'issues', 'discussed', 'here', 'should', 'be', 'addressed', 'there', 'need', 'to', 'be']...
[LEMMATIZED ]: ['issue', 'discussed', 'should', 'addressed', 'need', 'priority', 'authority', 'country', 'probably', 'know', 'whats', 'important']...
Jumlah token sebelum: 55 → sesudah: 31

[TOKENS     ]: ['the', 'excuse', 'that', 'brazilian', 'municipalities', 'do', 'not', 'have', 'the', 'funds', 'to', 'handle']...
[LEMMATIZED ]: ['excuse', 'brazilian', 'municipality', 'fund', 'handle', 'public', 'service', 'request', 'hold', 'water', 'evident', 'fact']..

## 5. Representasi Teks

### a. TF-IDF (Term Frequency–Inverse Document Frequency) 

**TF-IDF** mengukur seberapa penting suatu kata dalam sebuah dokumen relatif terhadap seluruh korpus.
- Cocok sebagai baseline karena menangkap pola kata kunci yang sering muncul di tiap jenis fallacy.
- Menghasilkan sparse matrix berdimensi tinggi.

In [23]:
# TF-IDF Vectorization — optimized for fallacy detection
# sublinear_tf=True → log(1+tf) dampens high-frequency term dominance
# ngram_range=(1,3) → captures longer fallacy-indicating phrases like "always been done", "everyone knows that"
# min_df=2 → removes ultra-rare terms that cause memorization
# max_df=0.95 → removes terms appearing in >95% of docs (noise)
tfidf_vectorizer = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1, 3),
    sublinear_tf=True,
    min_df=2,
    max_df=0.95,
)
tfidf_matrix = tfidf_vectorizer.fit_transform(df['processed_text'])

print(f"TF-IDF Matrix Shape: {tfidf_matrix.shape}")
print(f"  → {tfidf_matrix.shape[0]} dokumen, {tfidf_matrix.shape[1]} fitur")

# Tampilkan top fitur TF-IDF untuk beberapa sampel
feature_names = tfidf_vectorizer.get_feature_names_out()
print(f"\nContoh 20 fitur (kata/bigram/trigram): {list(feature_names[:20])}")

# Konversi ke DataFrame untuk preview
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=feature_names
)

print(f"\n--- Preview TF-IDF Matrix (5 dokumen pertama, 10 fitur pertama) ---")
tfidf_df.iloc[:5, :10]

TF-IDF Matrix Shape: (5370, 8000)
  → 5370 dokumen, 8000 fitur

Contoh 20 fitur (kata/bigram/trigram): ['abandon', 'abandoned', 'abhorrent', 'ability', 'able', 'able access', 'able continue', 'able express', 'able get', 'able make', 'able say', 'able speak', 'able speak mind', 'able use', 'abroad', 'absolute', 'absolutely', 'absurd', 'abuse', 'abuse power']

--- Preview TF-IDF Matrix (5 dokumen pertama, 10 fitur pertama) ---


,abandon,abandoned,abhorrent,ability,able,able access,able continue,able express,able get,able make
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### b. Word2Vec
**Word2Vec** mempelajari embedding vektor untuk setiap kata berdasarkan konteks kemunculannya.
- Menangkap hubungan semantik antar kata (misal: "authority" dekat dengan "expert", "tradition" dekat dengan "culture").
- Representasi dokumen diperoleh dengan merata-ratakan vektor semua kata dalam dokumen.
- Menghasilkan dense vector berdimensi tetap untuk setiap dokumen.

In [24]:
# Train Word2Vec pada corpus lemmatized
w2v_model = Word2Vec(
    sentences=df['lemmatized'].tolist(),
    vector_size=100,    # dimensi embedding
    window=5,           # context window
    min_count=2,        # abaikan kata yang muncul < 2 kali
    workers=4,
    epochs=20,
    sg=1                # skip-gram (lebih baik untuk dataset kecil)
)

print(f"Vocabulary size: {len(w2v_model.wv)}")
print(f"Vector dimension: {w2v_model.wv.vector_size}")

# Contoh kata-kata mirip
for word in ['government', 'freedom', 'tradition']:
    if word in w2v_model.wv:
        similar = w2v_model.wv.most_similar(word, topn=5)
        print(f"\nKata mirip dengan '{word}':")
        for w, score in similar:
            print(f"  {w}: {score:.4f}")

Vocabulary size: 6654
Vector dimension: 100

Kata mirip dengan 'government':
  outrageous: 0.6048
  subjugated: 0.6001
  relented: 0.5991
  federal: 0.5965
  ousted: 0.5936

Kata mirip dengan 'freedom':
  expression: 0.6827
  speech: 0.6332
  free: 0.6091
  inviolate: 0.6035
  evaporates: 0.5871

Kata mirip dengan 'tradition':
  deity: 0.5547
  patrimony: 0.5171
  habit: 0.5153
  historic: 0.5132
  worship: 0.5124


In [25]:
def document_vector(tokens, model):
    """Hitung rata-rata Word2Vec vector untuk seluruh token dalam dokumen."""
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    if len(vectors) == 0:
        return np.zeros(model.wv.vector_size)
    return np.mean(vectors, axis=0)

# Buat document vectors untuk semua komentar
w2v_vectors = np.array([
    document_vector(tokens, w2v_model) for tokens in df['lemmatized']
])

w2v_df = pd.DataFrame(
    w2v_vectors,
    columns=[f'w2v_dim_{i}' for i in range(w2v_model.wv.vector_size)]
)

print(f"Word2Vec Document Matrix Shape: {w2v_vectors.shape}")
print(f"  → {w2v_vectors.shape[0]} dokumen, {w2v_vectors.shape[1]} dimensi")
print(f"\n--- Preview Word2Vec Document Vectors (5 dokumen pertama, 10 dimensi pertama) ---")
w2v_df.iloc[:5, :10]

Word2Vec Document Matrix Shape: (5370, 100)
  → 5370 dokumen, 100 dimensi

--- Preview Word2Vec Document Vectors (5 dokumen pertama, 10 dimensi pertama) ---


,w2v_dim_0,w2v_dim_1,w2v_dim_2,w2v_dim_3,w2v_dim_4,w2v_dim_5,w2v_dim_6,w2v_dim_7,w2v_dim_8,w2v_dim_9
0,-0.143179,0.313680,0.061348,0.061057,0.061044,-0.283043,0.211651,0.403834,-0.127040,-0.028193
1,-0.142015,0.150530,0.101732,0.247849,-0.012118,-0.391233,0.087562,0.485941,-0.224959,-0.138929
2,-0.343969,0.254713,0.086639,0.009687,0.165461,-0.264466,0.027275,0.498432,-0.152881,-0.008060
3,-0.132087,0.084221,0.089468,0.065473,0.124449,-0.396054,0.090310,0.503461,-0.193612,-0.097347
4,-0.164386,0.321390,-0.038631,-0.023006,0.074467,-0.378160,0.076532,0.504369,-0.206425,-0.133870


## 6. Label Encoding

In [9]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['fallacy'])

print(f"Jumlah kelas: {len(label_encoder.classes_)}")
print(f"\nMapping label:")
for i, cls in enumerate(label_encoder.classes_):
    count = (df['label'] == i).sum()
    print(f"  {i} → {cls} ({count} sampel)")

Jumlah kelas: 9

Mapping label:
  0 → appeal to authority (350 sampel)
  1 → appeal to majority (383 sampel)
  2 → appeal to nature (412 sampel)
  3 → appeal to tradition (401 sampel)
  4 → appeal to worse problems (421 sampel)
  5 → false dilemma (391 sampel)
  6 → hasty generalization (379 sampel)
  7 → none (2202 sampel)
  8 → slippery slope (431 sampel)
